# SunPy Warm-up: SDO/AIA 태양 영상 분석

## 1. 연구 질문

SunPy를 이용해 SDO/AIA 관측 영상을 불러오고, 관측 데이터의 기본 정보와 태양 영상을 확인할 수 있는가?

## 2. 데이터 출처

- 관측 위성: Solar Dynamics Observatory (SDO)
- 관측 장비: Atmospheric Imaging Assembly (AIA)
- 데이터 접근 및 분석: SunPy

## 3. 데이터 다운로드

## 4. 전처리

## 5. 시각화

## 6. 관찰 결과

## 7. 남은 질문

### 1-1. 물리량에 단위 붙이기

천문 관측에서는 거리, 시간, 각도처럼 서로 다른 물리량을 다룬다.
`astropy.units`를 사용하면 숫자에 단위를 포함시키고 안전하게 단위를 변환할 수 있다.

In [1]:
import astropy.units as u

distance = 1500 * u.km
time_interval = 5 * u.s

speed = distance / time_interval

print("거리:", distance)
print("시간:", time_interval)
print("속력:", speed)

거리: 1500.0 km
시간: 5.0 s
속력: 300.0 km / s


In [2]:
speed_m_per_s = speed.to(u.m / u.s)

print("km/s 단위:", speed)
print("m/s 단위:", speed_m_per_s)

km/s 단위: 300.0 km / s
m/s 단위: 300000.0 m / s


In [3]:
print("객체:", distance)
print("숫자 부분:", distance.value)
print("단위 부분:", distance.unit)
print("객체의 자료형:", type(distance))

객체: 1500.0 km
숫자 부분: 1500.0
단위 부분: km
객체의 자료형: <class 'astropy.units.quantity.Quantity'>


### 1-2. 천문 관측 시각 다루기

`astropy.time.Time`을 사용하면 관측 시각을 여러 천문학적 시간 형식으로 표현하고,
두 관측 시각 사이의 간격도 계산할 수 있다.

In [4]:
from astropy.time import Time

observation_time = Time(
    "2024-05-10T12:30:00",
    format="isot",
    scale="utc"
)

print("관측 시각:", observation_time)
print("자료형:", type(observation_time))
print("입력 형식:", observation_time.format)
print("시간 척도:", observation_time.scale)

관측 시각: 2024-05-10T12:30:00.000
자료형: <class 'astropy.time.core.Time'>
입력 형식: isot
시간 척도: utc


- format="isot": 시간을 YYYY-MM-DDTHH:MM:SS 형태로 입력했다는 뜻
- scale="utc": 이 시각이 UTC 기준이라는 뜻

In [5]:
print("ISOT:", observation_time.isot)
print("ISO:", observation_time.iso)
print("Julian Date:", observation_time.jd)
print("Modified Julian Date:", observation_time.mjd)

ISOT: 2024-05-10T12:30:00.000
ISO: 2024-05-10 12:30:00.000
Julian Date: 2460441.0208333335
Modified Julian Date: 60440.520833333336


- 'isot': 날짜와 시간 사이에 T가 들어감
- 'iso': 날짜와 시간 사이에 공백이 들어감
- 'jd': 율리우스일(Julian Date)
- 'mjd': 수정 율리우스일(Modified Julian Date)

표현되는 숫자는 달라도 전부 동일한 순간을 나타낸다.

In [7]:
start_time = Time("2024-05-10T12:30:00", scale="utc")
end_time = Time("2024-05-10T12:42:30", scale="utc")

time_difference = end_time - start_time

print("시간 차이 객체:", time_difference)
print("초 단위:", time_difference.to_value("s"))
print("분 단위:", time_difference.to_value("min"))

시간 차이 객체: 0.00868055555555547
초 단위: 749.9999999999925
분 단위: 12.499999999999876


In [8]:
import astropy.units as u

next_time = start_time + 30 * u.min

print("시작 시각:", start_time.isot)
print("30분 후:", next_time.isot)

시작 시각: 2024-05-10T12:30:00.000
30분 후: 2024-05-10T13:00:00.000


### 1-3. 각도 단위와 픽셀 이해하기

`degree`와 `arcsec`는 하늘에서의 각도 또는 겉보기 크기를 나타낸다.
반면 `pixel`은 디지털 영상에서의 위치나 길이를 나타낸다.

픽셀을 각도로 변환하려면 영상의 공간 척도(spatial scale)가 필요하다.

In [9]:
import astropy.units as u

angle_deg = 1 * u.deg
angle_arcsec = angle_deg.to(u.arcsec)

print("각도:", angle_deg)
print("arcsec 변환:", angle_arcsec)

각도: 1.0 deg
arcsec 변환: 3600.0 arcsec


In [10]:
print((1 * u.deg).to(u.arcmin))
print((1 * u.arcmin).to(u.arcsec))
print((1 * u.arcsec).to(u.deg))

60.0 arcmin
60.0 arcsec
0.0002777777777777778 deg


In [11]:
solar_diameter = 0.5 * u.deg

print("degree:", solar_diameter)
print("arcsec:", solar_diameter.to(u.arcsec))

degree: 0.5 deg
arcsec: 1800.0 arcsec


pixel은 각도 단위가 아니라 영상 격자의 한 칸을 의미.

예를 들어 이미지 크기가 4096 × 4096 pixel이라면 가로와 세로에 각각 4096개의 영상 칸이 있다는 뜻.
그러나 이것만으로는 한 픽셀이 실제 하늘에서 몇 arcsec인지는 알 수 없다.

그 관계를 알려주는 것이 픽셀 스케일.

In [12]:
pixel_scale = 0.6 * u.arcsec / u.pixel

print("픽셀 스케일:", pixel_scale)

픽셀 스케일: 0.6 arcsec / pix


In [13]:
pixel_length = 100 * u.pixel
angular_length = pixel_length * pixel_scale

print("영상 위 길이:", pixel_length)
print("하늘에서의 각도:", angular_length)

영상 위 길이: 100.0 pix
하늘에서의 각도: 60.0 arcsec


In [14]:
angular_size = 300 * u.arcsec
pixel_size = angular_size / pixel_scale

print("각크기:", angular_size)
print("픽셀 수:", pixel_size)

각크기: 300.0 arcsec
픽셀 수: 500.0 pix


In [17]:
#(100 * u.pixel).to(u.arcsec)
(100 * u.pixel * pixel_scale).to(u.arcsec)

<Quantity 60. arcsec>

### 1-4. Helioprojective 좌표계 이해하기

Helioprojective Cartesian(HPC) 좌표계는 **특정 관측자가 바라본 태양의 겉보기 위치**를 2차원 평면 위에 나타내는 좌표계이다.

태양 원반 중심을 원점 `(0, 0)`으로 두고, 영상 위의 위치를 태양 중심으로부터 떨어진 각거리로 표현한다.

- `Tx`: 태양 중심으로부터 좌우 방향의 각거리
- `Ty`: 태양 중심으로부터 상하 방향의 각거리
- 주로 사용하는 단위: `arcsec`
- 원점 `(0, 0)`: 관측자에게 보이는 태양 원반의 중심

#### 좌표의 예시

| Helioprojective 좌표 | 태양 영상에서의 위치 |
|---|---|
| `(0, 0) arcsec` | 태양 원반 중심 |
| `(500, 0) arcsec` | 중심에서 오른쪽으로 500 arcsec |
| `(-500, 0) arcsec` | 중심에서 왼쪽으로 500 arcsec |
| `(0, 300) arcsec` | 중심에서 위쪽으로 300 arcsec |
| `(500, 200) arcsec` | 중심에서 오른쪽으로 500 arcsec, 위쪽으로 200 arcsec |

Helioprojective 좌표는 태양 표면에 고정된 위도와 경도가 아니다. 관측자의 시선 방향으로 투영된 **겉보기 좌표**이므로, 같은 태양 현상이라도 관측자의 위치와 관측 시각에 따라 좌표가 달라질 수 있다.

따라서 Helioprojective 좌표를 정의할 때는 다음 정보가 필요하다.

- `Tx`, `Ty`: 태양 원반 중심으로부터의 각거리
- `observer`: 태양을 관측한 위치
- `obstime`: 태양을 관측한 시각

#### Pixel 좌표와의 차이

- `pixel` 좌표는 영상 배열에서 몇 번째 칸인지를 나타낸다.
- Helioprojective 좌표는 태양 중심에서 얼마나 떨어져 보이는지를 각도로 나타낸다.
- SunPy의 `Map`은 관측 데이터의 메타데이터를 사용하여 pixel 좌표와 Helioprojective 좌표를 서로 변환한다.

> Helioprojective 좌표는 특정 시각에 특정 관측자가 바라본 태양 영상 위의 위치를, 태양 원반 중심으로부터의 좌우·상하 각거리 `(Tx, Ty)`로 나타내는 좌표계이다.

In [19]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from sunpy.coordinates import frames

solar_coordinate = SkyCoord(
    500 * u.arcsec,
    200 * u.arcsec,
    frame=frames.Helioprojective,
    observer="earth",
    obstime="2024-05-10T12:30:00"
)

print(solar_coordinate)
print("Tx:", solar_coordinate.Tx)
print("Ty:", solar_coordinate.Ty)

<SkyCoord (Helioprojective: obstime=2024-05-10T12:30:00.000, rsun=695700.0 km, observer=<HeliographicStonyhurst Coordinate for 'earth'>): (Tx, Ty) in arcsec
    (500., 200.)>
Tx: 500 arcsec
Ty: 200 arcsec


### 1-5. 태양 원반 중심과 limb의 좌표 확인하기

Helioprojective 좌표계에서 태양 원반 중심의 좌표는 `(Tx, Ty) = (0, 0)`이다.

태양의 **limb**는 관측자에게 보이는 태양 원반의 가장자리이다.  
Limb는 하나의 점이 아니라, 태양 중심으로부터 겉보기 태양 반지름만큼 떨어진 좌표들의 집합이다.

관측자와 관측 시각이 정해지면 태양의 겉보기 각반지름을 계산할 수 있다.

- 원반 중심: `(0, 0) arcsec`
- 서쪽 limb: `(R, 0) arcsec`
- 동쪽 limb: `(-R, 0) arcsec`
- 북쪽 limb: `(0, R) arcsec`
- 남쪽 limb: `(0, -R) arcsec`

여기서 `R`은 해당 시각과 관측자 위치에서 측정한 태양의 겉보기 각반지름이다.

태양과 지구 사이의 거리가 변하기 때문에 `R`은 항상 같은 값이 아니며, 지구에서는 대략 `960 arcsec`이다.

In [20]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from sunpy.coordinates import frames

observation_time = "2024-05-10T12:30:00"

solar_center = SkyCoord(
    0 * u.arcsec,
    0 * u.arcsec,
    frame=frames.Helioprojective,
    observer="earth",
    obstime=observation_time
)

print("태양 중심:", solar_center)
print("Tx:", solar_center.Tx)
print("Ty:", solar_center.Ty)

태양 중심: <SkyCoord (Helioprojective: obstime=2024-05-10T12:30:00.000, rsun=695700.0 km, observer=<HeliographicStonyhurst Coordinate for 'earth'>): (Tx, Ty) in arcsec
    (0., 0.)>
Tx: 0 arcsec
Ty: 0 arcsec


In [21]:
solar_angular_radius = solar_center.frame.angular_radius

print("태양의 겉보기 각반지름:", solar_angular_radius)
print("arcsec 단위:", solar_angular_radius.to(u.arcsec))

태양의 겉보기 각반지름: 949.846 arcsec
arcsec 단위: 949.846 arcsec


In [22]:
R = solar_angular_radius.to(u.arcsec)

west_limb = SkyCoord(
    R, 0 * u.arcsec,
    frame=solar_center.frame
)

east_limb = SkyCoord(
    -R, 0 * u.arcsec,
    frame=solar_center.frame
)

north_limb = SkyCoord(
    0 * u.arcsec, R,
    frame=solar_center.frame
)

south_limb = SkyCoord(
    0 * u.arcsec, -R,
    frame=solar_center.frame
)

print("원반 중심:", solar_center.Tx, solar_center.Ty)
print("서쪽 limb:", west_limb.Tx, west_limb.Ty)
print("동쪽 limb:", east_limb.Tx, east_limb.Ty)
print("북쪽 limb:", north_limb.Tx, north_limb.Ty)
print("남쪽 limb:", south_limb.Tx, south_limb.Ty)

원반 중심: 0 arcsec 0 arcsec
서쪽 limb: 949.846 arcsec 0 arcsec
동쪽 limb: -949.846 arcsec 0 arcsec
북쪽 limb: 0 arcsec 949.846 arcsec
남쪽 limb: 0 arcsec -949.846 arcsec


In [23]:
import numpy as np

limb_distance = np.sqrt(
    west_limb.Tx**2 + west_limb.Ty**2
)

print("서쪽 limb의 중심으로부터 각거리:", limb_distance)
print("태양의 각반지름:", R)

서쪽 limb의 중심으로부터 각거리: 949.8455051891756 arcsec
태양의 각반지름: 949.846 arcsec
